# Парсер академии селлеров Lamoda

Сбор статей публичной базы знаний https://academy.lamoda.ru/ и сохранение их в
Markdown (только текст). Ноутбук запускается сверху вниз в Google Colab без
ручных правок.

**Что делает парсер:**
1. Обходит дерево навигации (`/articles/...`) — `sitemap.xml` у сайта нет.
2. Отличает листовые статьи от страниц-разделов.
3. Извлекает тело статьи из server-HTML (JS-рендеринг не нужен — контент в HTML).
4. Конвертирует в Markdown и сохраняет `.md` + сводный `articles.jsonl`.

Парсер не падает на одной битой странице: каждый URL обрабатывается изолированно,
с таймаутами, ретраями и валидацией результата.


## 1. Установка зависимостей

In [ ]:
%pip install -q httpx selectolax markdownify nest_asyncio
print("deps installed")

## 2. Конфигурация

Селекторы вынесены в одно место — при смене вёрстки правится только здесь.

In [ ]:
import asyncio, json, re, hashlib, time, random
from pathlib import Path
from urllib.parse import urljoin, urlsplit, urlunsplit

import httpx
import nest_asyncio
from selectolax.parser import HTMLParser
from markdownify import markdownify as md_convert

nest_asyncio.apply()

BASE_URL    = "https://academy.lamoda.ru/"
ARTICLES_RE = "/articles/"

SEL_BODY        = "div.article-detail__text"
SEL_TITLE       = "h1"
SEL_DATE        = ".article-detail-date__value"
SEL_BREADCRUMB  = ".breadcrumb-item"
SEL_FEEDBACK    = ".article-feedback"

CONCURRENCY   = 1
TIMEOUT       = 20.0
MAX_RETRIES   = 3
BACKOFF_CAP   = 5.0
REQUEST_DELAY = 0.3
MIN_BODY_LEN  = 80
USER_AGENT    = "Mozilla/5.0 (compatible; LamodaAcademyParser/1.0)"
MAX_PAGES     = None

OUT_DIR   = Path("data/articles"); OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = Path("data/cache");    CACHE_DIR.mkdir(parents=True, exist_ok=True)
JSONL     = Path("data/articles.jsonl")

## 3. HTTP-слой: загрузка с кешем, таймаутами и ретраями

Сырой HTML кешируется на диск по хешу URL: повторный прогон ноутбука не нагружает
сайт и проходит за секунды. Ошибки сети не роняют прогон — возвращается `None`.

In [ ]:
def _cache_path(url: str) -> Path:
    h = hashlib.sha1(url.encode("utf-8")).hexdigest()[:16]
    return CACHE_DIR / f"{h}.html"

async def fetch(client: httpx.AsyncClient, url: str) -> str | None:
    """Возвращает HTML или None. Никогда не бросает исключение наружу."""
    cp = _cache_path(url)
    if cp.exists():
        return cp.read_text(encoding="utf-8")

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = await client.get(url)
            if r.status_code == 200:
                cp.write_text(r.text, encoding="utf-8")
                return r.text
            if r.status_code in (429, 500, 502, 503, 504):
                await asyncio.sleep(min(2 ** attempt, BACKOFF_CAP) + random.random())
                continue
            print(f"  skip {r.status_code}: {url}")
            return None
        except (httpx.HTTPError, asyncio.TimeoutError) as e:
            if attempt == MAX_RETRIES:
                print(f"  fail after {attempt}: {url} ({type(e).__name__})")
                return None
            await asyncio.sleep(min(2 ** attempt, BACKOFF_CAP) + random.random())
    return None

## 4. Нормализация и сбор ссылок

Дерево статей встроено в сайдбар каждой страницы, поэтому отдельная карта сайта не нужна — собираем ссылки на лету.

In [ ]:
def normalize(url: str) -> str:
    """Абсолютный URL без query/fragment, с единым trailing-slash."""
    absu = urljoin(BASE_URL, url)
    parts = urlsplit(absu)
    if parts.netloc != urlsplit(BASE_URL).netloc:
        return ""
    path = parts.path
    if not path.endswith("/"):
        path += "/"
    return urlunsplit((parts.scheme, parts.netloc, path, "", ""))

def extract_article_links(tree: HTMLParser) -> set[str]:
    links = set()
    for a in tree.css("a[href]"):
        href = a.attributes.get("href") or ""
        if ARTICLES_RE in href:
            n = normalize(href)
            if n:
                links.add(n)
    return links

## 5. Классификация и извлечение

Листовая статья = есть непустой контейнер `article-detail__text` **и** блок
обратной связи `.article-feedback`. Страница-раздел этого тела не имеет и служит
только источником новых ссылок.

In [ ]:
def to_markdown(body_html: str) -> str:
    """HTML-фрагмент тела -> Markdown (только текст, без картинок)."""
    md = md_convert(body_html, heading_style="ATX", strip=["img", "figure", "picture"])
    md = re.sub(r"\]\((/[^)]*)\)", lambda m: "](" + urljoin(BASE_URL, m.group(1)) + ")", md)
    md = re.sub(r"\n{3,}", "\n\n", md).strip()
    return md

def parse_article(url: str, html: str) -> dict | None:
    """Возвращает статью dict или None, если страница не листовая/пустая."""
    tree = HTMLParser(html)

    body = tree.css_first(SEL_BODY)
    feedback = tree.css_first(SEL_FEEDBACK)
    if body is None or feedback is None:
        return None

    title_node = tree.css_first(SEL_TITLE)
    title = title_node.text(strip=True) if title_node else ""

    body_md = to_markdown(body.html or "")
    if not title or len(body_md) < MIN_BODY_LEN:
        return None

    date_node = tree.css_first(SEL_DATE)
    updated = date_node.text(strip=True) if date_node else ""

    crumbs = [c.text(strip=True).lstrip("/").strip() for c in tree.css(SEL_BREADCRUMB)]
    crumbs = [c for c in crumbs if c and c != "\u2026"]

    slug = urlsplit(url).path.strip("/").replace("/", "__")
    return {"url": url, "title": title, "breadcrumbs": crumbs,
            "updated": updated, "slug": slug, "markdown": body_md}

## 6. Сохранение в Markdown

Каждая статья — отдельный `.md` с YAML-фронтматтером (источник, заголовок, путь, дата).

In [ ]:
def save_markdown(art: dict) -> Path:
    fm = [
        "---",
        f"url: {art['url']}",
        f"title: {json.dumps(art['title'], ensure_ascii=False)}",
        f"breadcrumbs: {json.dumps(art['breadcrumbs'], ensure_ascii=False)}",
        f"updated: {art['updated']}",
        "---",
        "",
        f"# {art['title']}",
        "",
        art["markdown"],
        "",
    ]
    path = OUT_DIR / f"{art['slug']}.md"
    path.write_text("\n".join(fm), encoding="utf-8")
    return path

## 7. Обход (BFS) и оркестрация

Стартуем с главной, идём по найденным ссылкам `/articles/`. Каждый URL посещается
один раз (`visited`). Листовые статьи сохраняем, у разделов берём только ссылки.
Конкурентность ограничена семафором.

In [ ]:
async def crawl() -> dict:
    visited: set[str] = set()
    frontier: set[str] = {normalize(BASE_URL)}
    saved, skipped, failed = [], [], []

    sem = asyncio.Semaphore(CONCURRENCY)
    limits = httpx.Limits(max_connections=CONCURRENCY, max_keepalive_connections=CONCURRENCY)

    async with httpx.AsyncClient(headers={"User-Agent": USER_AGENT},
                                 timeout=TIMEOUT, follow_redirects=True,
                                 limits=limits, http2=False) as client:

        def progress():
            done = len(saved) + len(skipped) + len(failed)
            print(f"\r  обработано {done}/{len(visited)} | "
                  f"сохранено {len(saved)} | пропущено {len(skipped)} | "
                  f"упало {len(failed)}", end="", flush=True)

        async def handle(url: str):
            async with sem:
                await asyncio.sleep(REQUEST_DELAY + random.uniform(0, 0.2))
                html = await fetch(client, url)
            if html is None:
                failed.append(url); progress()
                return set()
            try:
                tree = HTMLParser(html)
                new_links = extract_article_links(tree)
                art = parse_article(url, html)
                if art is not None:
                    save_markdown(art)
                    saved.append(art)
                else:
                    skipped.append(url)
                progress()
                return new_links
            except Exception as e:
                print(f"\n  parse error: {url} ({type(e).__name__}: {e})")
                failed.append(url); progress()
                return set()

        level = 0
        while frontier:
            batch = [u for u in frontier if u and u not in visited]
            if MAX_PAGES:
                room = MAX_PAGES - len(visited)
                if room <= 0:
                    break
                batch = batch[:room]
            visited.update(batch)
            frontier = set()
            level += 1
            print(f"\n[уровень {level}] в очереди {len(batch)} URL")
            results = await asyncio.gather(*(handle(u) for u in batch))
            for links in results:
                frontier |= {l for l in links if l not in visited}
        print()

    with JSONL.open("w", encoding="utf-8") as f:
        for a in saved:
            f.write(json.dumps(a, ensure_ascii=False) + "\n")

    return {"visited": len(visited), "saved": len(saved),
            "skipped": len(skipped), "failed": failed}

## 8. Запуск

In [ ]:
t0 = time.time()
report = asyncio.run(crawl())
print(f"\nГотово за {time.time() - t0:.1f}s")
print(f"  посещено URL : {report['visited']}")
print(f"  сохранено    : {report['saved']} статей -> {OUT_DIR}/")
print(f"  пропущено    : {report['skipped']} (разделы/пустые)")
print(f"  упало        : {len(report['failed'])}")
if report["failed"]:
    print("  проблемные URL:")
    for u in report["failed"][:20]:
        print("   -", u)

## 9. Проверка результата

In [ ]:
mds = sorted(OUT_DIR.glob("*.md"))
print(f"Файлов .md: {len(mds)}\n")
if mds:
    sample = mds[0]
    print(f"--- {sample.name} ---")
    print(sample.read_text(encoding="utf-8")[:1200])